# Ingesta y verificación de datos — Entrega 1

Proyecto Integrador de Ciencia de Datos (UTN FRM 2026): **¿qué combinación de nivel de ELO, apertura jugada, modalidad de ritmo y color de piezas predice mejor quién gana una partida de ajedrez online, y qué tan larga es?**

**Unidad de análisis**: una partida de ajedrez individual jugada en Lichess.

**Fuente**: API pública de [Lichess](https://lichess.org) (`GET /api/games/user/{username}`), sin necesidad de autenticación. Se descargan partidas reales de una lista de jugadores activos en distintos rangos de ELO (desde nivel club hasta top mundial), para que el rating tenga variación genuina.

Este notebook **no reimplementa la lógica del pipeline**: importa las clases de `src/` y las ejecuta acá, con narrativa explicando cada decisión. El pipeline también puede correrse fuera del notebook con `python -m src.pipeline`.

In [ ]:
# Imports and base paths.
# The notebook's working directory is switched to the repo root so that the
# relative paths in config.yaml (data/raw, data/processed) resolve the same
# way whether the pipeline runs from the notebook or from `python -m src.pipeline`.
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path("..").resolve()
if Path.cwd() != REPO_ROOT:
    os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from src.clean_data import DataCleaner
from src.download_data import DataDownloader
from src.feature_engineering import FeatureEngineer
from src.utils import load_config

CONFIG_PATH = Path("config/config.yaml")
config = load_config(CONFIG_PATH)

## Sección 1 — Descarga automatizada

`DataDownloader` pide a `GET /api/games/user/{username}` las partidas rated de cada jugador configurado (`config.yaml -> lichess.usernames`), en formato PGN, con apertura (`opening=true`) incluida. La descarga es idempotente: si el archivo ya existe en `data/raw/`, se saltea.

In [ ]:
# Download PGN games for every configured user (skips automatically if already present)
downloader = DataDownloader(config)
pgn_paths = downloader.download_all()
pgn_paths

## Sección 2 — Parseo de PGN

El formato PGN no es un CSV: cada partida es un bloque de encabezados `[Clave "Valor"]` seguido de la lista de jugadas en notación algebraica. `DataCleaner.parse_pgn_file` separa el archivo en bloques por partida y extrae los encabezados relevantes (ELO, resultado, apertura, control de tiempo, terminación) con una regex — no hace falta `python-chess` porque este proyecto no analiza la calidad de las jugadas, solo metadata de la partida.

In [ ]:
# Parse, clean and validate all downloaded PGNs
cleaner = DataCleaner(config)
df, raw_row_count = cleaner.clean(pgn_paths)
df = cleaner.optimize_dtypes(df)

print(f"Partidas crudas: {raw_row_count:,}")
print(f"Partidas válidas: {len(df):,} ({100 * len(df) / raw_row_count:.1f}% retenidas)")
df.head()

## Sección 3 — Ingeniería de features

`FeatureEngineer.transform` agrega, todo vectorizado:

- `diferencia_elo`, `elo_promedio`, `favorito`: quién entraba como favorito según el rating.
- `nivel_promedio`: banda de ELO de la partida (principiante a top mundial).
- `modalidad`: bullet / blitz / rapid / clásica, según el tiempo estimado de partida (tiempo base + 40 × incremento, la convención de Lichess).
- `es_sorpresa`: 1 si ganó el jugador con menor ELO. **Target de clasificación adicional.**
- `familia_apertura`: agrupación de la apertura jugada por su letra ECO (A-E).

In [ ]:
# Feature engineering
engineer = FeatureEngineer(config)
df = engineer.transform(df)
df.head()

## Sección 4 — Exportación

Se guarda el dataset final en Parquet (compresión `snappy`) más un sample en CSV para inspección rápida.

In [ ]:
# Export processed dataset
processed_dir = Path(config["paths"]["processed_dir"])
processed_dir.mkdir(parents=True, exist_ok=True)

clean_parquet_path = Path(config["paths"]["clean_parquet"])
df.to_parquet(clean_parquet_path, engine="pyarrow", compression="snappy", index=False)

sample_path = Path(config["paths"]["clean_sample_csv"])
sample_size = min(config["processing"]["sample_size"], len(df))
df.sample(n=sample_size, random_state=config["processing"]["random_state"]).to_csv(
    sample_path, index=False
)

print(f"Parquet: {clean_parquet_path}")
print(f"Sample CSV ({sample_size} filas): {sample_path}")

## Sección 5 — Verificación

Se recarga el Parquet recién escrito (no el `df` en memoria) para confirmar que lo que quedó persistido en disco es correcto de punta a punta.

In [ ]:
# Reload the persisted parquet to verify it independently of the in-memory df
df_check = pd.read_parquet(clean_parquet_path)
df_check.info()

In [ ]:
df_check.describe()

In [ ]:
# Null check: ECO/Opening can be null only if the download skipped opening=true;
# everything else should be complete after filter_invalid_rows.
nulls = df_check.isna().sum()
nulls[nulls > 0]

In [ ]:
# Target variable distributions
print("resultado:")
print(df_check["resultado"].value_counts(normalize=True))
print("\nmodalidad:")
print(df_check["modalidad"].value_counts())
print("\nnivel_promedio:")
print(df_check["nivel_promedio"].value_counts())
print(f"\nTasa de sorpresas (gana el de menor ELO): {100 * df_check['es_sorpresa'].mean():.1f}%")

In [ ]:
# Primer vistazo a la pregunta de investigación: probabilidad de sorpresa
# según qué tan grande es la diferencia de ELO entre los jugadores.
df_check["abs_diferencia_elo"] = df_check["diferencia_elo"].abs()
bins = pd.cut(df_check["abs_diferencia_elo"], bins=[0, 50, 100, 200, 400, 5000])
tasa_sorpresa_por_brecha = df_check.groupby(bins, observed=True)["es_sorpresa"].mean()
print(tasa_sorpresa_por_brecha)

fig, ax = plt.subplots(figsize=(7, 4))
tasa_sorpresa_por_brecha.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_ylabel("Tasa de sorpresa")
ax.set_xlabel("Diferencia de ELO absoluta")
ax.set_title("¿Cuánto importa la diferencia de ELO para que gane el 'underdog'?")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Cantidad de jugadas por modalidad — sanity check geoespacial-temporal
fig, ax = plt.subplots(figsize=(7, 4))
df_check.boxplot(column="cantidad_jugadas", by="modalidad", ax=ax)
ax.set_ylabel("Cantidad de jugadas (plies)")
ax.set_title("Duración de la partida por modalidad de ritmo")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Final integrity checks
assert df_check.shape[0] > 100, "El volumen final es muy bajo"
assert df_check["resultado"].isin(["Gana Blancas", "Gana Negras", "Empate"]).all(), "resultado tiene valores inesperados"
assert df_check["cantidad_jugadas"].min() > 0, "Toda partida válida debe tener al menos 1 jugada"
assert df_check[["WhiteElo", "BlackElo"]].isna().sum().sum() == 0, "No debe haber nulos en ELO"
assert df_check["es_sorpresa"].isin([0, 1]).all(), "es_sorpresa debe ser binaria"

print(f"Shape final: {df_check.shape}")
print("Todas las verificaciones pasaron correctamente.")